In [0]:
%sql
SELECT * FROM PARQUET.`/path/to/file` LIMIT 10; -- read directly from file
CREATE TABLE table_name AS SELECT * FROM default.temp; -- CTAS statement to create table automaticaly ifer schema
CREATE OR REPLACE TEMP VIEW sales_view AS SELECT * FROM csv.`/path/to/file`; -- CTAS statement to create view from csv file

CREATE OR REPLACE TEMP VIEW sales_view
  USING DELTA AS 
        SELECT * FROM read_files('patj/to/file', format => 'csv', header => true, sep => '|', mode='FAILFAST'); -- CTAS with custom options

DESCRIBE EXTENDED sales_view; -- describe view column/data type + metadata

# Catalogs and Schemas

In [0]:
DESCRIBE CATALOG `${catalog_name}`; -- Catalog Name, Comment, Owner, Type
DESCRIBE SCHEMA default; -- Namespace, Comment, Owner, Location

# Load Incrementaly

In [0]:
CREATE TABLE table_name; -- create empt6y table without schema
COPY INTO table_name FROM '/path/to/file' FILEFORMAT = parquet COPY_OPTIONS(mergeSchema => 'true') -- schema provided by source

CREATE TABLE sales_csv -- create external table
(order_id LONG, desc STRING, transaction TIMESTAMP, item STRING) -- provide schema
 USING CSV OPTIONS (header => 'true', sep => '|', delimiter => '\t') LOCATION '/path/to/file'; -- add options

# Buil-in Methods

In [0]:
SELECT *, 
current_timestamp() AS updated,
input_file_name() AS file_name,
CAST(CAST(current_timestamp()/1e6 AS TIMESTAMP) AS DATE) AS date
FROM CSV.`/path/to/file`;

# Basic Transformations

In [ ]:
%sql

CREATE OR REPLACE TABLE sales_csv DEEP CLONE sales_arhive_csv; -- copy data and metadata from table physically
CREATE OR REPLACE TABLE sales_csv SHALLOW CLONE sales_arhive_csv; -- copy only metadata from table
INSERT OVERWRITE sales_csv SELECT * FROM sales_csv; -- overwrite table with new data TRUNCATE + INSERT, do not apply new schema
CREATE OR REPLACE TABLE sales_csv AS SELECT * FROM sales_csv; -- overwrite table with new data and apply new schema

MERGE target_table AS target
USING source_table AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *; -- INSERT + UPDATE + JOIN in one SQL query

MERGE user_table AS target
USING (SELECT * FROM user_table WHERE user_id = 1) AS source
ON target.user_id = source.user_id
WHEN MATCHED AND target.email IS NULL AND source.email IS NOT NULL -- additional condition
THEN UPDATE SET email = source.email
WHEN NOT MATCHED THEN
 INSERT (id, name, email, phone)
  VALUES (source.id, source.name, source.email, source.phone);

MERGE user_table AS target
USING (SELECT * FROM user_table WHERE user_id = 1) AS source
ON target.user_id = source.user_id AND target.timestamp = source.timestamp
WHEN MATCHED THEN UPDATE SET *; -- UPDATE all columns only

CREATE TABLE purhcase_orders (
  id STRING,
  transaction_timestamp STRING,
  price STRING,
  date DATE GENERATED ALWAYS AS (CAST(CAST(transaction_timestamp/1e6 AS TIMESTAMP) AS DATE) -- generated column
  COMMENT 'generated based on transaction_timestamp column') -- column comment
);

# Add table constraint

In [0]:
%sql

ALTER TABLE user_profile ADD CONSTRAINT date_check CHECK (date >= '2020-01-01'); -- check constraint
ALTER TABLE user_profile ADD CONSTRAINT name_check CHECK (name IS NOT NULL); -- not null constraint
ALTER TABLE user_profile ADD CONSTRAINT email_check CHECK (email IS NOT NULL AND email LIKE '%@%.%'); -- email constraint by pattern

-- constrains can be showed at DESCRIBE TABLE user_profile at TABLE PROPERTIES

# Cleaning Data

In [0]:
CREATE TABLE IF NOT EXISTS user_bronze (
  user_id STRING,
  user_first_touch_timestamp BIGINT,
  email STRING,
  updated TIMESTAMP,
  first_touch TIMESTAMP,
  first_touch_date DATE,
  first_touch_time STRING,
  email_domain STRING
);

CREATE OR REPLACE TABLE user_silver_working AS 
SELECT * FROM users_bronze;

INSERT OVERWRITE user_silver 
  SELECT DISTINCT(*) FROM user_silver_working; -- remove duplicates

INSERT OVERWRITE user_silver 
  SELECT (user_id, user_first_touch_timestamp, MAX(email) AS email, MAX(updated) AS updated)
   FROM user_silver_working GROUP BY user_id, user_first_touch_timestamp; -- deduplicate by specific columns

SELECT MAX(row_counts) <= 1 AS is_unique FROM (
  SELECT user_id, COUNT(*) AS row_counts FROM user_silver_working GROUP BY user_id); -- chech for duplicates by specific columns

INSERT INTO user_silver (
  SELECT *, 
  TO_DATE(DATE_FORMAT(first_touch, "yyy-M-d")) AS first_touch_date,
  DATE_FORMAT(first_touch, "HH:mm:ss") AS first_touch_time,
  REGEX_EXTRACT(email, "@(.*)", 0) AS email_domain
  FROM (
    SELECT *, CAST(user_first_touch_timestamp/1e6 AS TIMESTAMP) AS first_touch 
    FROM user_silver_working
  )
); -- using regex and datetime functions

# Complex Transformations

In [0]:
CREATE OR REPLACE TEMP VIEW event_streams AS 
SELECT STRING(key), STRING(value) FROM kafka_events_raw; -- kafka raw data

SELECT * FROM kafka_events_raw WHERE value:event_name = 'finalize' ORDER BY key LIMIT 1; -- working with nested value (: for json, . for STRUCT type)

SELECT schema_of_json('{"a": 1, "b": "some_text') AS schema; -- parse json and store schema

CREATE OR REPLACE TEMP VIEW parsed_events AS SELECT json.* FROM (
  SELECT from_json(value, 'STRUCT<event_name:STRING, event_data:STRING>') AS json 
  FROM event_streams); -- parse json and store as STRUCT type
  
CREATE OR REPLACE TEMP VIEW exploded_events AS SELECT *, EXPLODE(items) as item FROM parsed_events; -- explode separate array into multiple rows
SELECT * FROM exploded_events WHERE SIZE(item) > 2; -- size provides count of array elements

-- Nesting functions
SELECT user_id, 
COLLECT_SET(items.item_id) AS cart_history -- collect all item_id in uniq set
FROM exploded_events GROUP BY user_id ORDER BY user_id;

SELECT user_id, 
FLATTEN(COLLECT_SET(items.item_id)) AS cart_history -- create one array from set of arrays
FROM exploded_events GROUP BY user_id ORDER BY user_id; 

SELECT user_id, 
COLLECT_SET(event_name) as event_history,
ARRAY_DISTINCT(FLATTEN(COLLECT_SET(items.item_id))) AS cart_history -- create one array from set of arrays with uniq values
FROM exploded_events GROUP BY user_id ORDER BY user_id; 

-- Pivot table
SELECT * FROM item_purchase PIVOT(SUM(item.quantity) FOR item_id IN ('P_FOAM_K'));

# SQL UDF

In [0]:
CREATE OR REPLACE FUNCTION sale_announcement(item_name STRING, item_price INT)
RETURNS STRING
RETURN CONCAT("The", item_name, " is on sale for $", ROUND(item_price * 0.8, 2));

SELECT *, sale_announcement(name, price) -- function can be used in SQL
 FROM item_lookup;

DESCRIBE FUNCTION EXTENDED sale_announcement; -- show detailed info about function

SELECT current_catalog(); -- show current catalog

# Advance DeltaLake Features

In [0]:
-- liquid clustering
CREATE OR REPLACE TABLE events_liquid CLUSTER BY (user_id) AS SELECT * FROM events; -- create new table with clustering
ALTER TABLE events CLUSTER BY (user_id); -- change current table clustering
OPTIMIZE events; -- reccommends scheduling optimize on clustered tables ~ 1-2h
DESCRIBE HISTORY events; -- show history of changes mand on table
RESTORE TABLE students TO VERSION AS OF 1; -- restore table to 1 version
VACUUM events RETAIN 10; -- remove old versions of table (10 days by default)